# Python + AI APIs


In [1]:
!pip install openai

  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached openai-2.38.0-py3-none-any.whl (1.3 MB)
   ---------------------------------------- 0.0/200.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/200.6 kB ? eta -:--:--
   -- ------------------------------------- 10.2/200.6 kB ? eta -:--:--
   ------------- ------------------------- 71.7/200.6 kB 777.7 kB/s eta 0:00:01
   ---------------------------------------- 200.6/200.6 kB 1.7 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.30.0 requires packaging<24,>=16.8, but you have packaging 24.2 which is incompatible.


## 1. Connecting to the API
We use the `openai` client. 
- **For OpenAI**, you need an API key.
- **For Ollama**, you just point the `base_url` to your local machine (usually `http://localhost:11434/v1`).

In [3]:
from openai import OpenAI
import os


from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="gemma2:2b",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! 👋 

What can I do for you today? 😄  



##  2. Response Handling


In [5]:
response = client.chat.completions.create(
    model="gemma2:2b",
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ],
    temperature=0.3
)


print("--- RAW RESPONSE OBJECT ---")
print(response)
print("\n" + "-"*40 + "\n")


answer = response.choices[0].message.content
print("--- EXTRACTED ANSWER ---")
print(answer)

--- RAW RESPONSE OBJECT ---
ChatCompletion(id='chatcmpl-59', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris. 🇫🇷 \n', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1779876598, model='gemma2:2b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=13, prompt_tokens=21, total_tokens=34, completion_tokens_details=None, prompt_tokens_details=None))

----------------------------------------

--- EXTRACTED ANSWER ---
The capital of France is Paris. 🇫🇷 



## 3. Streaming Outputs

In [6]:
import sys

print("Streaming response: ", end="")

stream = client.chat.completions.create(
    model="gemma2:2b",
    messages=[
        {"role": "user", "content": "Write a short haiku about coding."}
    ],
    stream=True  # This is the magic parameter!
)

for chunk in stream:
    # Extract the token from the chunk
    if chunk.choices[0].delta.content is not None:
        word = chunk.choices[0].delta.content
        print(word, end="")
        sys.stdout.flush() # Force it to print immediately

Streaming response: Lines of code take flight,
Logic dance in the dark net,
Dream app takes form now. 


## 4. Prompt Chaining


In [8]:
messy_data = """
Hey guys, so John Smith is coming to the meeting at 3. Oh, and bring Sarah Jenkins too. 
I think Mike T. might be late because of traffic.
"""

# --- Step 1: Extraction ---
prompt_1 = f"Extract only the names of the people from this text. Return them as a raw list.\n\n{messy_data}"

response_1 = client.chat.completions.create(
    model="gemma2:2b",
    messages=[{"role": "user", "content": prompt_1}]
)
extracted_names = response_1.choices[0].message.content
print("--- Output of Step 1 ---")
print(extracted_names)

# --- Step 2: Formatting ---
prompt_2 = f"Take these names and put them into a clean, comma-separated list:\n\n{extracted_names}"

response_2 = client.chat.completions.create(
    model="gemma2:2b",
    messages=[{"role": "user", "content": prompt_2}]
)
final_formatted_names = response_2.choices[0].message.content
print("\n--- Output of Step 2 ---")
print(final_formatted_names)

--- Output of Step 1 ---
[John Smith, Sarah Jenkins, Mike T.] 


--- Output of Step 2 ---
John Smith, Sarah Jenkins, Mike T. 

